## 🎯 ML Business Problem Statement
> The goal of this project is to predict salary trends in government jobs using features such as education level, sector, and role category.

- The model helps estimate expected salary ranges for government jobs based on role and qualification patterns. This can support career planning, job market analysis, and data-driven decision making for job seekers and career platforms.
----
### End Clients for Project

**1️⃣ Job Seekers / Students**
- estimate salary expectations
- compare sectors
- understand role trends

**2️⃣ Career Guidance Platforms**
- suggest realistic salary ranges
- guide users toward sectors

**3️⃣ Recruitment Analytics Teams**
- analyze salary structures
- study market patterns

**4️⃣ Educational/Career Institutions**
- understand demand trends
- align training with job market

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [97]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression,SGDRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from xgboost import XGBRegressor

In [85]:
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error, root_mean_squared_error,r2_score # Huber Loss , R^2 method    
from sklearn.preprocessing import LabelEncoder,OneHotEncoder,OrdinalEncoder


from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [17]:
from sklearn.model_selection import train_test_split

### Data Collection

In [45]:
df = pd.read_csv("../Data/Job_Data_Trimmed.csv")

In [46]:
df.head()

,Job Type,salary_minimum,salary_maximum,salary_band,salary_range,qualification_level,sector,role_category
0,Government,63000.0,63000.0,Medium,0.0,undergraduate,Banking,Officer
1,Government,25000.0,25000.0,Low,0.0,undergraduate,Government General,Associate
2,Government,30000.0,60000.0,Medium,30000.0,undergraduate,PSU,Other
3,Government,57700.0,182400.0,Medium,124700.0,postgraduate,Government Admin,Academic
4,Government,25000.0,25000.0,Low,0.0,undergraduate,PSU,Engineer


### Feature Selection And Preprocessing
----
As of the Problem Statement is regarding the Regression 

- We choose Specific Feature for the Pattern recognition
- Selected Features are
    - Role Category
    - Qualification Level
    - Sector 

In [47]:
feature = df[["role_category","qualification_level","sector","salary_minimum"]]

In [48]:
feature.isna().sum()

role_category          0
qualification_level    0
sector                 0
salary_minimum         0
dtype: int64

In [49]:
feature["salary_minimum"].skew()

np.float64(1.569524950165759)

#### Spliting and Encoding Flow 
----
1. Separate X and y
2. Train-test split
3. Fit encoders/scalers ONLY on training data
4. Transform training data
5. Transform test data using fitted transformers

In [65]:
# 1. Seperate the X and y
X = df[["qualification_level", "sector", "role_category"]]
y = df["salary_minimum"]

# 2. Splitting data
X_train, X_val, y_train, y_val = train_test_split(X, y, train_size = 0.8,test_size=0.2, random_state=42)

In [66]:
X_train["qualification_level"].unique()

array(['undergraduate', 'postgraduate', 'diploma', 'doctorate', 'other',
       'school_level'], dtype=object)

##### As of Data in the Target column is not in the Normal Distribution we use some trasfromation techniques
- We used to do the seperate for y_train, y_test to reduce the data leakage
- But it didn't have much impact on the target column so we proceed without any trasformation to target column.

In [34]:
# y_train.plot(kind="density")
# y_val.plot(kind="density") 
# plt.show()

In [35]:
# y_train = np.log1p(y_train)
# y_val = np.log1p(y_val)

In [36]:
# y_train.plot(kind="density")
# y_val.plot(kind="density") 
# plt.show()

In [32]:
y_train.skew()

np.float64(-0.8703763720775994)

In [33]:
y_val.skew()

np.float64(-0.9035357885518903)

### Model Training

In [67]:
ordinal_cols = ["qualification_level"]
nominal_cols = ["sector","role_category"]

In [76]:
# 3. Encoding using the transformer

preprocessor = ColumnTransformer(transformers=[("qual_encoder", OrdinalEncoder(), ordinal_cols),
                                               ("onehot", OneHotEncoder(handle_unknown="ignore"), nominal_cols)])

#### Linear regression

In [77]:
# 4. Intializing the Linear regression model into the pipeline

# 1. Linear Regression 
pipeline = Pipeline(steps=[("preprocessing", preprocessor),
                           ("model", LinearRegression())])



In [78]:
pipeline.fit(X_train, y_train)

,steps,"[('preprocessing', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('qual_encoder', ...), ('onehot', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


### Predictions

In [88]:
train_pred = pipeline.predict(X_train)
test_pred = pipeline.predict(X_test)

### Validation 

In [92]:
# Validation Data
mae = mean_absolute_error(y_test, test_pred)
mse = mean_squared_error(y_test, test_pred)
rmse = mse ** 0.5

r2 = r2_score(y_test, y_pred)

print("Validation Data loss")
print("--"*20)
print("MAE :", mae)
print("RMSE :", rmse)
print("R2 Score :", r2)
print()
# training data
mae_t = mean_absolute_error(y_train, train_pred)
mse_t = mean_squared_error(y_train, train_pred)
rmse_t = mse_t ** 0.5

r2_t = r2_score(y_test, y_pred)

print("Training Data loss")
print("--"*20)
print("MAE :", mae_t)
print("RMSE :", rmse_t)
print("R2 Score :", r2_t)

Validation Data loss
----------------------------------------
MAE : 25602.184267931887
RMSE : 34839.81208541035
R2 Score : 0.06191801827436416

Training Data loss
----------------------------------------
MAE : 24423.505057287624
RMSE : 33775.77322489008
R2 Score : 0.06191801827436416


#### After Trying with Linear Regression
- Linear Regression was used as a baseline model, but the low R² score indicated that salary patterns were not strongly linear. This suggested the presence of more complex relationships between categorical features and salary, motivating the use of more advanced nonlinear models.
- Here we can say model is underftting clearly

#### Random Forest Regressor

#### Training Model

In [93]:
rf_pipeline = Pipeline([("preprocessing", preprocessor),
                        ("model", RandomForestRegressor(n_estimators=100,random_state=42))])

In [94]:
rf_pipeline.fit(X_train, y_train)

,steps,"[('preprocessing', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('qual_encoder', ...), ('onehot', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [95]:
rf_pred = rf_pipeline.predict(X_test)
r2_score(y_test, rf_pred)

-0.03688354792201021

#### Validation 

In [96]:
# Validation Data
val_pred = rf_pipeline.predict(X_test)

mae = mean_absolute_error(y_test, val_pred)
mse = mean_squared_error(y_test, val_pred)
rmse = mse ** 0.5

r2 = r2_score(y_test, val_pred)

print("Validation Data loss")
print("--"*20)
print("MAE :", mae)
print("RMSE :", rmse)
print("R2 Score :", r2)
print()



# training data
train_pred = rf_pipeline.predict(X_train)

mae_t = mean_absolute_error(y_train, train_pred)
mse_t = mean_squared_error(y_train, train_pred)
rmse_t = mse_t ** 0.5

r2_t = r2_score(y_test, y_pred)

print("Training Data loss")
print("--"*20)
print("MAE :", mae_t)
print("RMSE :", rmse_t)
print("R2 Score :", r2_t)

Validation Data loss
----------------------------------------
MAE : 26798.395089901744
RMSE : 36628.60666083759
R2 Score : -0.03688354792201021

Training Data loss
----------------------------------------
MAE : 20842.558213123375
RMSE : 30310.906323157807
R2 Score : 0.06191801827436416


#### Decsision Tree Regressor

#### Training Model

In [98]:
dt_pipeline = Pipeline([("preprocessing", preprocessor),
                        ("model", DecisionTreeRegressor(max_depth = 14,random_state=42))])

In [99]:
dt_pipeline.fit(X_train, y_train)

,steps,"[('preprocessing', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('qual_encoder', ...), ('onehot', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [100]:
dt_pred = dt_pipeline.predict(X_test)
r2_score(y_test, dt_pred)

-0.3024358954247881

#### Validation 

In [101]:
# Validation Data
val_pred = dt_pipeline.predict(X_test)

mae = mean_absolute_error(y_test, val_pred)
mse = mean_squared_error(y_test, val_pred)
rmse = mse ** 0.5

r2 = r2_score(y_test, val_pred)

print("Validation Data loss")
print("--"*20)
print("MAE :", mae)
print("RMSE :", rmse)
print("R2 Score :", r2)
print()



# training data
train_pred = dt_pipeline.predict(X_train)

mae_t = mean_absolute_error(y_train, train_pred)
mse_t = mean_squared_error(y_train, train_pred)
rmse_t = mse_t ** 0.5

r2_t = r2_score(y_test, y_pred)

print("Training Data loss")
print("--"*20)
print("MAE :", mae_t)
print("RMSE :", rmse_t)
print("R2 Score :", r2_t)

Validation Data loss
----------------------------------------
MAE : 29363.92375778523
RMSE : 41051.930523391
R2 Score : -0.3024358954247881

Training Data loss
----------------------------------------
MAE : 20001.17490900401
RMSE : 30280.84147668396
R2 Score : 0.06191801827436416
